In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from sentence_transformers import SentenceTransformer

from utils import (
    build_binary_relevance_maps,
    build_eval_items,
    build_global_corpus,
    evaluate_sentence_transformer,
    evaluate_tfidf,
    load_triplets,
)

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = os.path.join(PROJECT_ROOT, "dataset", "hard_triplets_tfidf_wikidata.json")
TOPKS = [3, 5, 10, 20]
BATCH = 32

QWEN_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Loaded {len(raw_data)} items")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval anchors with non-empty positives: {len(eval_items)}")

Loaded 186 items
Global corpus size: 2229
Eval anchors with non-empty positives: 186


In [3]:
evaluate_sentence_transformer(
    QWEN_MODEL_NAME,
    "Baseline: Qwen3-Embedding",
    eval_items,
    global_corpus,
    relevance_scores_per_item,
    TOPKS,
    batch_size=BATCH,
)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5243.79it/s]



Baseline: Qwen3-Embedding
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0699  0.0144  0.0225  0.0797  0.1022    0.0699  0.0686
@5     0.0720  0.0242  0.0340  0.0835  0.1183    0.0720  0.0714
@10    0.0731  0.0499  0.0551  0.0989  0.2366    0.0731  0.0762
@20    0.0468  0.0655  0.0505  0.1039  0.3118    0.0468  0.0655
